In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import scipy as sp
import pandas as pd
import geopandas as gpd
import fiona
from shapely.geometry import Polygon
from scipy import sparse
import xarray as xr
import geopandas
import contextily as cx
import colorcet as cc

fiona.drvsupport.supported_drivers["KML"] = "rw"

#### Area

In [ ]:
# Intersted Area
out_name = "partition_output"
us = gpd.read_file("./Texas.kml", driver="KML")
already_ran = False

#### Emissions

In [ ]:
prior_pth = "./jacobian_runs_Texas_2020/Test_IMI_0000/OutputDir/HEMCO_diagnostics.202001010000.nc"
prior_emis = xr.load_dataset(prior_pth)
# prior_emis['lon'][:],prior_emis['lat'][:] #0.625*0.5
prior_emis

<xarray.Dataset>
Dimensions:              (lon: 33, lat: 32, lev: 47, time: 1)
Coordinates:
  * lon                  (lon) float64 -108.8 -108.1 -107.5 ... -89.38 -88.75
  * lat                  (lat) float64 23.0 23.5 24.0 24.5 ... 37.5 38.0 38.5
  * lev                  (lev) float64 0.9925 0.9775 ... 0.0001387 3.8e-05
  * time                 (time) datetime64[ns] 2020-01-01
Data variables: (12/19)
    hyam                 (lev) float64 2.402 332.1 986.4 ... 41.41 13.87 3.8
    hybm                 (lev) float64 0.9925 0.9742 0.9526 ... 0.0 0.0 0.0
    P0                   float64 1e+05
    AREA                 (lat, lon) float64 3.557e+09 3.557e+09 ... 3.024e+09
    EmisCH4_SoilAbsorb   (time, lat, lon) float32 0.0 0.0 ... -1.822e-12
    EmisCH4_Termites     (time, lat, lon) float32 4.371e-14 ... 3.238e-12
    ...                   ...
    EmisCH4_Landfills    (time, lat, lon) float32 0.0 0.0 0.0 ... 0.0 2.323e-11
    EmisCH4_Livestock    (time, lat, lon) float32 0.0 0.0 ... 1.889e-11
    EmisCH4_Coal         (time, lat, lon) float32 0.0 0.0 ... 8.695e-12
    EmisCH4_Gas          (time, lat, lon) float32 0.0 0.0 ... 4.887e-12
    EmisCH4_Oil          (time, lat, lon) float32 0.0 0.0 ... 1.299e-11
    EmisCH4_Total        (time, lat, lon) float32 4.371e-14 ... 7.744e-11
Attributes:
    title:        ./OutputDir/HEMCO  diagnostics
    history:      Created by routine NC_CREATE (in ncdf_mod.F90)
    format:       NetCDF-4
    conventions:  COARDS
    reference:    http://wiki.geos-chem.org/The_HEMCO_Users_Guide
    contact:      GEOS-Chem Support Team (geos-chem-support@as.harvard.edu)

In [ ]:
res_pth = "./integrated_methane_inversion/imi_output_Texas_dir_2020/Test_IMI/inversion/gridded_posterior.nc"
posterior_emis = xr.load_dataset(res_pth)
# posterior_emis['lon'][:],posterior_emis['lat'][:] #0.625*0.5
posterior_emis

<xarray.Dataset>
Dimensions:      (lat: 33, lon: 34)
Coordinates:
  * lon          (lon) float32 -108.8 -108.1 -107.5 ... -89.38 -88.75 -88.12
  * lat          (lat) float32 23.0 23.5 24.0 24.5 25.0 ... 37.5 38.0 38.5 39.0
Data variables:
    ScaleFactor  (lat, lon) float64 2.285 2.285 2.285 ... 2.744 2.744 2.744
    S_post       (lat, lon) float64 0.0212 0.0212 0.0212 ... 0.1553 0.1553
    A            (lat, lon) float64 0.9152 0.9152 0.9152 ... 0.379 0.379 0.379

In [ ]:
ori_res_pth = "./integrated_methane_inversion/imi_output_Texas_dir_2020/Test_IMI/inversion/inversion_result.nc"
ori_posterior_emis = xr.load_dataset(ori_res_pth)
# posterior_emis['lon'][:],posterior_emis['lat'][:] #0.625*0.5
ori_posterior_emis

<xarray.Dataset>
Dimensions:      (nvar: 242)
Dimensions without coordinates: nvar
Data variables:
    KTinvSoK     (nvar, nvar) float32 0.1441 0.08852 0.03797 ... 8.363 47.6
    KTinvSoyKxA  (nvar) float32 5.715 6.642 5.364 7.555 ... 732.8 275.8 139.8
    ratio        (nvar) float32 0.1869 -0.02109 -0.1228 ... 1.904 -0.6453 1.285
    xhat         (nvar) float32 1.187 0.9789 0.8772 ... 2.904 0.3547 2.285
    S_post       (nvar, nvar) float32 0.2421 -0.004012 ... -0.002229 0.0212
    A            (nvar, nvar) float32 0.03164 0.01605 ... 0.008915 0.9152

#### Parameters

In [8]:
n_elements = 242
xhat_flux = posterior_emis["ScaleFactor"][:]  # posterior scaling factors
shat = posterior_emis["S_post"][:]  # posterior error covariance matrix
xa1 = prior_emis["EmisCH4_Total"][0, :]  # prior in units kg m-2 s-1
hem_grid = xa1.copy()
AK = posterior_emis["A"][:]
cllon = posterior_emis["lon"][:]
cllat = posterior_emis["lat"][:]

SaCAP = np.zeros(n_elements)  # prior covariance matrix - in scaling factor space
SaCAP.fill(0.5**2)
SaCAP = np.diag(SaCAP)

In [ ]:
statevector_file = (
    "./integrated_methane_inversion/imi_output_Texas_dir_2020/Test_IMI/StateVector.nc"
)
statevector = xr.load_dataset(statevector_file)


def do_gridding(vector, statevector):
    """
    Project input vector onto the inversion grid using information from the state vector file.
    Input vector should be a numpy array of scale factors (SF), diagonal elements of the posterior
    error covariance matrix (S_post), or diagonal elements of the averaging kernel matrix (A).
    """

    # Map the input vector (e.g., scale factors) to the state vector grid
    nlat = len(statevector["lat"])
    nlon = len(statevector["lon"])
    target_array = np.empty(statevector["StateVector"].shape)
    target_array[:] = np.nan
    for ilat in range(nlat):
        for ilon in range(nlon):
            element_id = statevector["StateVector"].values[ilat, ilon]
            if ~np.isnan(element_id):
                target_array[ilat, ilon] = vector[int(element_id) - 1]

    # Convert to data array
    lat = statevector["lat"].values
    lon = statevector["lon"].values
    target_array = xr.DataArray(
        target_array, [("lat", list(lat)), ("lon", list(lon))], attrs={"units": "none"}
    )
    print(target_array.shape)

    return target_array


gridded_SaCAP = do_gridding(np.diagonal(SaCAP), statevector)

(33, 34)


In [ ]:
# xa_grid.shape,gridded_SaCAP.shape,saa.shape,xhat_flux.shape,shh.shape

((32, 33), (33, 34), (32, 33), (33, 34), (32, 33))

#### RUN INVERSION

##### Parameters Pre-process

In [10]:
### Get fluxes in right units
iscale = 1
xa_grid = xa1  # Prior flux
xh_grid = xa_grid * xhat_flux  # Posterior flux
saa = xa_grid * gridded_SaCAP  # Prior flux error covariance
# shh = xa_grid * shat #Posterior flux error covariance
shh = xh_grid * shat  # Posterior flux error covariance

In [12]:
def get_raster_list(fgrid):

    flon_un = np.sort(np.array(list(set(fgrid.lon.tolist()))))
    flat_un = np.sort(np.array(list(set(fgrid.lat.tolist()))))

    # Get resolution of gridded product
    xdiff = np.abs(np.mean(np.diff(flon_un))) / 2
    ydiff = np.abs(np.mean(np.diff(flat_un))) / 2

    # Make a geopandas polygon for each grid
    flist = []
    for idx in range(len(fgrid)):

        # Get lat/lon center
        xcent = fgrid.iloc[idx].lon
        ycent = fgrid.iloc[idx].lat

        # Make polygon
        lon_point_list = [xcent - xdiff, xcent - xdiff, xcent + xdiff, xcent + xdiff]
        lat_point_list = [ycent - ydiff, ycent + ydiff, ycent + ydiff, ycent - ydiff]

        polygon_geom = Polygon(zip(lon_point_list, lat_point_list))
        # polygon = gpd.GeoDataFrame(index=[0], crs="EPSG:4326", geometry=[polygon_geom])

        flist.append(polygon_geom)
        # print(idx+1, 'of', len(fgrid))
    return flist


tgrid = pd.DataFrame(
    np.array([(x, y) for y in cllat[:-1] for x in cllon[:-1]]), columns=["lon", "lat"]
)
tgrid["ind"] = np.arange(tgrid.shape[0])
tesp = gpd.GeoDataFrame(tgrid, geometry=gpd.points_from_xy(tgrid.lon, tgrid.lat))
tesp = tesp.set_crs("EPSG:4326")
# tsub = gpd.clip(tesp, us)
# tsel = tsub.ind
# tgrid_sel = tgrid.iloc[tsel]
tlist = get_raster_list(tesp)


# Very fast function to get area of a grid cell
def area_of_grid(x, y):
    return (
        (np.pi / 180)
        * (6371**2)
        * abs(np.sin(0.0174533 * y[0]) - np.sin(0.0174533 * y[2]))
        * np.abs(x[0] - x[2])
    )


# Get area of grid cells
def area_cells(ifgrid):
    x, y = ifgrid.exterior.coords.xy
    iarea = area_of_grid(x, y) * 1000 * 1000
    return iarea


tareas = np.array([area_cells(q) for q in tlist]).reshape(32, 33)

In [13]:
xa_f = xa_grid * 3600 * tareas * iscale  # convert to fluxes
Sa_f = sparse.csc_matrix(saa * 3600 * iscale * tareas)
S_hat_f = sparse.csc_matrix(shh * 3600 * iscale * tareas)
x_hat_f = xh_grid * 3600 * tareas * iscale

In [14]:
xa_f.shape, Sa_f.shape, S_hat_f.shape, x_hat_f.shape

((32, 33), (32, 33), (32, 33), (32, 33))

In [15]:
# ['EmisCH4_SoilAbsorb','EmisCH4_Termites','EmisCH4_Lakes','EmisCH4_Seeps','EmisCH4_Wetlands','EmisCH4_BiomassBurn',
#  'EmisCH4_OtherAnth','EmisCH4_Rice','EmisCH4_Wastewater','EmisCH4_Landfills','EmisCH4_Livestock','EmisCH4_Coal','EmisCH4_Gas','EmisCH4_Oil']
prior_wetlands = prior_emis["EmisCH4_Wetlands"].isel(time=0)
prior_livestock = prior_emis["EmisCH4_Livestock"].isel(time=0)
prior_oil = prior_emis["EmisCH4_Oil"].isel(time=0)
prior_gas = xr.load_dataset(prior_pth)["EmisCH4_Gas"].isel(time=0)
prior_landfills = prior_emis["EmisCH4_Landfills"].isel(time=0)
prior_coal = prior_emis["EmisCH4_Coal"].isel(time=0)
prior_wastewater = prior_emis["EmisCH4_Wastewater"].isel(time=0)
prior_otherna = (
    prior_emis["EmisCH4_BiomassBurn"].isel(time=0)
    + prior_emis["EmisCH4_Termites"].isel(time=0)
    + prior_emis["EmisCH4_Seeps"].isel(time=0)
)
prior_otheranth = prior_emis["EmisCH4_OtherAnth"].isel(time=0)
prior_soilabs = prior_emis["EmisCH4_SoilAbsorb"].isel(time=0)
prior_rice = prior_emis["EmisCH4_Rice"].isel(time=0)
prior_lake = prior_emis["EmisCH4_Lakes"].isel(time=0)

In [82]:
### Get emissions in right units
# base units in kg/h, but include scaling factor if want to use something else
iscale = 1

# Make emissions state vector and error covariance
# z_a = np.stack([prior_wetlands*tareas, prior_livestock*tareas, prior_oil*tareas, prior_gas*tareas, prior_rice*tareas, prior_landfills*tareas, \
#                       prior_coal*tareas, prior_wastewater*tareas, prior_otherna*tareas, prior_otheranth*tareas, prior_lake*tareas], axis=0) * 3600* iscale
z_a = (
    np.stack(
        [
            prior_wetlands,
            prior_livestock,
            prior_oil,
            prior_gas,
            prior_rice,
            prior_landfills,
            prior_coal,
            prior_wastewater,
            prior_otherna,
            prior_otheranth,
            prior_lake,
        ],
        axis=0,
    )
    * 1e9
    * 3600
    * iscale
)
other_fac = 0.1
# Sz = sparse.diags(((np.stack([prior_wetlands*tareas*gridded_SaCAP, prior_livestock*tareas*gridded_SaCAP, prior_oil*tareas*gridded_SaCAP, prior_gas*tareas*gridded_SaCAP, prior_rice*tareas*gridded_SaCAP, prior_landfills*tareas*gridded_SaCAP, \
#                       prior_coal*tareas*gridded_SaCAP, prior_wastewater*tareas*gridded_SaCAP, prior_otherna*tareas*gridded_SaCAP, prior_otheranth*tareas*gridded_SaCAP, prior_lake*tareas*gridded_SaCAP], axis=0)*3600*iscale)).reshape(-1)).tocsc()
Sz = sparse.diags(
    (
        (
            np.stack(
                [
                    prior_wetlands * other_fac,
                    prior_livestock * other_fac,
                    prior_oil * other_fac,
                    prior_gas * other_fac,
                    prior_rice * other_fac,
                    prior_landfills * other_fac,
                    prior_coal * other_fac,
                    prior_wastewater * other_fac,
                    prior_otherna * other_fac,
                    prior_otheranth * other_fac,
                    prior_lake,
                ],
                axis=0,
            )
            * 1e9
            * 3600
            * iscale
        )
    ).reshape(-1)
).tocsc()
# Sz = sparse.diags(((z_a * other_fac * iscale)**2).reshape(-1)).tocsc()
z_a = z_a.reshape(-1)

In [83]:
prior_wetlands

<xarray.DataArray 'EmisCH4_Wetlands' (lat: 32, lon: 33)>
array([[0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 0.0000000e+00,
        0.0000000e+00, 0.0000000e+00],
       [0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 0.0000000e+00,
        0.0000000e+00, 0.0000000e+00],
       [0.0000000e+00, 0.0000000e+00, 0.0000000e+00, ..., 0.0000000e+00,
        0.0000000e+00, 0.0000000e+00],
       ...,
       [8.5742042e-13, 7.1864232e-13, 1.3077519e-12, ..., 5.5278278e-12,
        1.3702543e-11, 1.4715148e-11],
       [5.4848975e-13, 9.8879899e-13, 2.4065923e-12, ..., 5.6967543e-12,
        7.9450821e-12, 9.2828644e-12],
       [1.4373287e-13, 3.1899379e-12, 5.3568998e-12, ..., 5.9935923e-12,
        4.7974788e-12, 3.4060951e-12]], dtype=float32)
Coordinates:
  * lon      (lon) float64 -108.8 -108.1 -107.5 -106.9 ... -90.0 -89.38 -88.75
  * lat      (lat) float64 23.0 23.5 24.0 24.5 25.0 ... 36.5 37.0 37.5 38.0 38.5
    time     datetime64[ns] 2020-01-01
Attributes:
    long_name:         CH4_emissions_from_wetlands
    units:             kg/m2/s
    averaging_method:  mean

In [84]:
# Remove zero entries so that inverse can compute
x_zero = xa_f == 0

xa = np.array(xa_f)[~x_zero] * iscale
Sa = sparse.diags(np.array(saa)[~x_zero] * 3600 * tareas[~x_zero] * iscale).tocsc()
S_hat = sparse.diags(np.array(shh)[~x_zero] * 3600 * tareas[~x_zero] * iscale).tocsc()
# S_hat = np.array(S_hat_f)[~x_zero]
# S_hat = S_hat[:,~x_zero] #sparse.diags(shh[~x_zero] * 3600 * bareas[~x_zero]).tocsc()
x_hat = np.array(xh_grid)[~x_zero] * 3600 * tareas[~x_zero] * iscale

In [85]:
xa_f_a = np.array(xa_f).reshape(-1)
non_indexs = np.where(xa_f_a != 0)
full_vars = 11
Mbase = np.zeros((xa.shape[0], xa_f_a.shape[0]))
for i in range(Mbase.shape[0]):
    Mbase[i, non_indexs[0][i]] = 1
Mbase = sparse.csr_matrix(Mbase)

for idx in range(full_vars):
    if idx == 0:
        M = Mbase.copy()
    else:
        M = sparse.hstack([M, Mbase])

In [ ]:
# MAP solution for inversion - follows Equations 1 and 2 in manuscript
K = sparse.csr_matrix(M)

# Take out zeros from prior
xa_bad = (z_a == 0) | np.isnan(z_a)

# Take out zero columns from Jacobian
non_zero_inds = sparse.find(K)
non_zero_cols = np.array(list(set(non_zero_inds[1])))
bad_cols = ~np.isin(np.arange(K.shape[1]), non_zero_cols)

# Update matrices
psel = bad_cols | xa_bad
px, py = np.meshgrid(np.where(~psel)[0], np.where(~psel)[0])

# ...prior
zA = z_a[~psel]  # sector prior
ZA = Sz[px, py]  # sector prior covariance matrix
# ZA = sparse.csc_matrix(np.diag(ZA.diagonal()))

# ...jacobian
KK = K[:, ~psel]  # .tocsc()    # K

# Compute posterior error covariance
Sh_s = S_hat.tocsc()  # f1 posterior error covariance
Sa_s = Sa.tocsc()  # f1 prior error covariance

# Get their inverse
II = sparse.identity(Sa_s.shape[0]).tocsc()
Sa_s_1 = sparse.linalg.spsolve(Sa_s, II)  # f1 prior inverse
Sh_s_1 = sparse.linalg.spsolve(Sh_s, II)  # f1 posterior inverse
Sh_minus_Sa = Sh_s_1 - Sa_s_1

# Do posterior update explicitly if memory allows #(fomula 2)
II2 = sparse.identity(ZA.shape[0]).tocsc()
ZA_1 = sparse.linalg.spsolve(ZA.tocsc(), II2)
KSK = (KK.T.dot(Sh_minus_Sa)).dot(KK)
SZHAT_inv = KSK + ZA_1
SZHAT = sp.linalg.inv(SZHAT_inv.todense())
SZHAT = sparse.csc_matrix(SZHAT)

# If there's a memory crunch, use the equivalent form: Woodbury formula #Memory Limitation
# Sh_minus_Sa_1 = sparse.linalg.spsolve(Sh_s_1 - Sa_s_1, II)
# iterm1 = ZA.dot(KK.T)
# iterm2 = KK.dot(ZA)
# iterm3 = Sh_minus_Sa_1 + iterm2.dot(KK.T)
# iterm4 = sparse.linalg.inv(iterm3)
# iterm5 = iterm4.dot(iterm2)
# iterm6 = iterm1.dot(iterm5)
# SZHAT = ZA - iterm6

# Compute posterior emissions #(formula 1)
A = sparse.identity(Sh_s.shape[0]) - Sh_s.dot(Sa_s_1)
term1 = (SZHAT.dot(KK.T)).dot(Sh_s_1)
term2 = A.dot(xa - KK.dot(zA)) + (x_hat - xa)
zupdate = zA + term1.dot(term2)
zhat = np.zeros(len(z_a))
zhat[~psel] = zupdate

# Save posterior error covariance
Z_hat = np.zeros((len(z_a), len(z_a)))
Z_hat[px, py] = SZHAT.todense()

In [70]:
Sh_minus_Sa.todense().min()

-213.35953605793424

In [235]:
# Alternative solution - following Derivation 2 in text. User can verify that we get the same answer
if False:

    # Run Inverse using MAP equations
    # Get observational error covariance in prior space - this follows Derivation 2 in SI of the manuscript
    # You get exactly the same solution (within computer noise) following Equations 1 and 2
    Sy_p_inv = sparse.linalg.inv(S_hat) - sparse.linalg.inv(Sa)
    Sy_p = sparse.linalg.inv(Sy_p_inv)
    y_p = xa + Sy_p.dot(sparse.linalg.spsolve(S_hat, x_hat - xa))

    K = sparse.csr_matrix(M)

    # Take out zeros from prior
    xa_bad = (z_a == 0) | np.isnan(z_a)

    # Take out zero columns from Jacobian
    non_zero_inds = sparse.find(K)
    non_zero_cols = np.array(list(set(non_zero_inds[1])))
    bad_cols = ~np.isin(np.arange(K.shape[1]), non_zero_cols)

    # Take out outlier fluxes
    r_bad = np.array([False] * len(y_p))  # We are keeping them all here

    # Update matrices
    psel = bad_cols | xa_bad
    px, py = np.meshgrid(np.where(~psel)[0], np.where(~psel)[0])

    # ...prior
    zA = z_a[~psel]
    ZA = sparse.diags(Sz.diagonal()[~psel]).tocsc()

    # ...observations
    Y = y_p[~r_bad]
    RR = Sy_p[~r_bad, :]
    RR = RR[:, ~r_bad].tocsc()

    # ...jacobian
    KK = K[:, ~psel]
    KK = KK[~r_bad, :].tocsc()

    # Compute posterior error covariance
    # Do posterior update using Woodbury formulation
    # iterm1 = ZA.dot(KK.T)
    # iterm2 = KK.dot(ZA)
    # iterm3 = RR + iterm2.dot(KK.T)
    # iterm4 = sparse.linalg.inv(iterm3)
    # iterm5 = iterm4.dot(iterm2)
    # iterm6 = iterm1.dot(iterm5)
    # SZHAT = ZA - iterm6

    # Posterior covariance the traditional rodgers way
    II = sparse.csc_matrix(np.eye(ZA.shape[0]))
    Z_hat_1 = KK.T.dot(Sy_p_inv.dot(KK)) + sparse.linalg.spsolve(ZA, II)
    SZHAT = sparse.linalg.spsolve(Z_hat_1, II).tocsc()
    Z_hat = np.zeros((len(z_a), len(z_a)))
    Z_hat[px, py] = SZHAT.todense()

    # Posterior emissions
    zupdate = zA + SZHAT.dot(KK.T.dot(sparse.linalg.spsolve(RR, Y - KK.dot(zA))))
    zhat = np.zeros(len(z_a))
    zhat[~psel] = zupdate

#### Error summation

In [87]:
full_vars = [
    "wetlands",
    "livestock",
    "oil",
    "gas",
    "rice",
    "landfills",
    "coal",
    "wastewater",
    "otherna",
    "otheranth",
    "lake",
]
Nlen = 32 * 33

In [89]:
# ~~~~~Error summation - totals
indices_list = []
for i in range(len(full_vars)):
    indices_list.append(np.repeat(i, Nlen))
indices_list = np.concatenate(indices_list).ravel()


# Function to integrate error covariance based on integration operator h
def integrate_error(h):

    hZh = h.dot(Z_hat.dot(h.T))  # total posterior error
    hSh = h.dot(Sz.dot(h.T))  # total prior error

    return np.sqrt(hZh)[0][0], np.sqrt(hSh)[0][0]


# Get error for total budget and different sectors
h_total = np.ones((1, Z_hat.shape[1]))
tot_post_err, tot_prior_err = integrate_error(h_total)


# Get error for wetland
h_wet = np.reshape((indices_list == 0) * 1, (1, len(indices_list)))
wet_post_err, wet_prior_error = integrate_error(h_wet)

# Get error for livestock
h_livestock = np.reshape((indices_list == 1) * 1, (1, len(indices_list)))
livestock_post_err, livestock_prior_error = integrate_error(h_livestock)

# Get error for oil
h_oil = np.reshape((indices_list == 2) * 1, (1, len(indices_list)))
oil_post_err, oil_prior_error = integrate_error(h_oil)

# Get error for gas
h_gas = np.reshape((indices_list == 3) * 1, (1, len(indices_list)))
gas_post_err, gas_prior_error = integrate_error(h_gas)

# Get error for rice
h_rice = np.reshape((indices_list == 4) * 1, (1, len(indices_list)))
rice_post_err, rice_prior_error = integrate_error(h_rice)

# Get error for landfills
h_landfills = np.reshape((indices_list == 5) * 1, (1, len(indices_list)))
landfills_post_err, landfills_prior_error = integrate_error(h_landfills)

# Get error for coal
h_coal = np.reshape((indices_list == 6) * 1, (1, len(indices_list)))
coal_post_err, coal_prior_error = integrate_error(h_coal)

# Get error for waste
h_waste = np.reshape((indices_list == 7) * 1, (1, len(indices_list)))
waste_post_err, waste_prior_error = integrate_error(h_waste)

# Get error for other nature
h_otherna = np.reshape((indices_list == 8) * 1, (1, len(indices_list)))
otherna_post_err, otherna_prior_error = integrate_error(h_otherna)

# Get error for other anthro
h_otheranth = np.reshape((indices_list == 9) * 1, (1, len(indices_list)))
otheranth_post_err, otheranth_prior_error = integrate_error(h_otheranth)

# Get error for lake
h_lake = np.reshape((indices_list == 10) * 1, (1, len(indices_list)))
lake_post_err, lake_prior_error = integrate_error(h_lake)


# Plot result
sum_dict = {
    "wetland": [
        np.sum(zhat * h_wet),
        wet_post_err,
        np.sum(z_a * h_wet),
        wet_prior_error,
    ],
    "livestock": [
        np.sum(zhat * h_livestock),
        livestock_post_err,
        np.sum(z_a * h_livestock),
        livestock_prior_error,
    ],
    "oil": [np.sum(zhat * h_oil), oil_post_err, np.sum(z_a * h_oil), oil_prior_error],
    "gas": [np.sum(zhat * h_gas), gas_post_err, np.sum(z_a * h_gas), gas_prior_error],
    "rice": [
        np.sum(zhat * h_rice),
        rice_post_err,
        np.sum(z_a * h_rice),
        rice_prior_error,
    ],
    "landfills": [
        np.sum(zhat * h_landfills),
        landfills_post_err,
        np.sum(z_a * h_landfills),
        landfills_prior_error,
    ],
    "coal": [
        np.sum(zhat * h_coal),
        coal_post_err,
        np.sum(z_a * h_coal),
        coal_prior_error,
    ],
    "waste": [
        np.sum(zhat * h_waste),
        waste_post_err,
        np.sum(z_a * h_waste),
        waste_prior_error,
    ],
    "otherna": [
        np.sum(zhat * h_otherna),
        otherna_post_err,
        np.sum(z_a * h_otherna),
        otherna_prior_error,
    ],
    "h_otheranth": [
        np.sum(zhat * h_otheranth),
        otheranth_post_err,
        np.sum(z_a * h_otheranth),
        otheranth_prior_error,
    ],
    "lake": [
        np.sum(zhat * h_lake),
        lake_post_err,
        np.sum(z_a * h_lake),
        lake_prior_error,
    ],
}

sum_df = pd.DataFrame.from_dict(
    sum_dict, orient="index", columns=["zhat", "zhat_err", "za", "za_err"]
)

sum_df.loc["total"] = sum_df.sum(axis=0)
sum_df.loc["total", "zhat_err"] = np.sqrt(np.sum(sum_df.zhat_err.iloc[:-1] ** 2))
sum_df.loc["total", "za_err"] = np.sqrt(np.sum(sum_df.za_err.iloc[:-1] ** 2))


# What total flux within that box
flux_box = np.sum(x_hat)
prior_box = np.sum(xa)
h_flux = np.ones((1, S_hat.shape[0]))
flux_err = np.sqrt(h_flux.dot(S_hat.dot(h_flux.T)))[0][0]
prior_err = np.sqrt(h_flux.dot(Sa.dot(h_flux.T)))[0][0]

# Output the summary dataframe in Tg/yr
sum_df * 1e-9 * 24 * 365

,zhat,zhat_err,za,za_err
wetland,0.797754,0.000391,0.202715,0.000421
livestock,1.103714,0.000757,0.788702,0.000831
oil,0.603219,0.000581,0.382289,0.000579
gas,2.323269,0.000968,1.191096,0.001021
rice,0.111458,0.000219,0.054291,0.000218
landfills,0.826037,0.000551,0.374103,0.000572
coal,0.350280,0.000322,0.152168,0.000365
waste,0.139210,0.000210,0.053949,0.000217
otherna,0.094197,0.000287,0.103409,0.000301
h_otheranth,0.063902,0.000158,0.028717,0.000159


#### PLOT

In [ ]:
# full_vars = ['wetlands', 'livestock', 'oil', 'gas', 'rice', 'landfills', 'coal', 'wastewater', 'otherna', 'otheranth', 'lake']
idx = 0

print("Plotting sector:", full_vars[idx])
ind1 = idx * Nlen
ind2 = (idx + 1) * Nlen

# Sum emissions
xg = zhat[ind1:ind2]
pg = z_a[ind1:ind2]

# Grid emissions in each sector
ogrid = tgrid.copy()
ogrid["post"] = np.nan
ogrid["prior"] = np.nan
ogrid.loc[:, "post"] = xg
ogrid.loc[:, "prior"] = pg
xgrid = np.array(ogrid.post).reshape((len(cllat) - 1, len(cllon) - 1)) / tareas * 1e6
pgrid = np.array(ogrid.prior).reshape((len(cllat) - 1, len(cllon) - 1)) / tareas * 1e6

# Plot
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 15))

df = pd.DataFrame(
    {
        "Latitude": np.array([[v] * len(cllon[:-1]) for v in cllat[:-1]]).reshape(-1),
        "Longitude": list(cllon[:-1].values) * len(cllat[:-1]),
        "prior": pgrid.reshape(-1),
        "post": xgrid.reshape(-1),
    }
)

geo_df = geopandas.GeoDataFrame(
    df, geometry=geopandas.points_from_xy(df.Longitude, df.Latitude), crs="EPSG:4326"
)
points = geopandas.points_from_xy(df.Longitude, df.Latitude)
polygons = []
for point in points:
    x, y = point.x, point.y
    quad = Polygon([(x, y), (x, y + 0.5), (x + 0.625, y + 0.5), (x + 0.625, y)])
    polygons.append(quad)

geo_df["polygons"] = polygons
geo_df = geo_df.set_geometry("polygons")
df_wm = geo_df.to_crs(epsg=3857)

sm = plt.cm.ScalarMappable(
    cmap=cc.cm.linear_kryw_5_100_c67_r, norm=plt.Normalize(vmin=0, vmax=np.max(pgrid))
)
sm.set_array([])
# ax1 = df_wm.plot(column="prior", figsize=(10, 10), alpha=0.3,cmap=cc.cm.linear_kryw_5_100_c67_r,ax=ax1,vmin=0)
ax1 = df_wm.plot(
    column="prior",
    figsize=(10, 10),
    alpha=0.3,
    cmap="YlOrRd",
    ax=ax1,
    vmin=0,
    vmax=np.max(pgrid),
)
ax1.set_axis_off()
cx.add_basemap(ax1)

fig.colorbar(sm, ax=ax1, fraction=0.03)
ax1.set_title("Prior (kg km$^{-2}$ h$^{-1}$): " + full_vars[idx])

# ax2 = df_wm.plot(column="post", figsize=(10, 10), alpha=0.3,cmap=cc.cm.linear_kryw_5_100_c67_r,ax=ax2,vmin=0)
ax2 = df_wm.plot(
    column="post",
    figsize=(10, 10),
    alpha=0.3,
    cmap="YlOrRd",
    ax=ax2,
    vmin=0,
    vmax=np.max(pgrid),
)
ax2.set_axis_off()
cx.add_basemap(ax2)
fig.colorbar(sm, ax=ax2, fraction=0.03)
ax2.set_title("Posterior(kg km$^{-2}$ h$^{-1}$): " + full_vars[idx])

c = ax3.pcolormesh(cllon[:-1], cllat[:-1], pgrid, cmap="YlOrRd", vmin=0)
# c = ax3.pcolormesh(cllon[:-1], cllat[:-1], pgrid/tareas*1e6, cmap='YlOrRd', vmin=0, vmax=np.max(xgrid))
fig.colorbar(c, ax=ax3, fraction=0.03)
us.plot(ax=ax3, facecolor="none", edgecolor="k", linewidth=3)
ax3.set_xlim((-107, -90))
ax3.set_ylim((24.5, 37.5))
ax3.set_title("Prior (kg km$^{-2}$ h$^{-1}$): " + full_vars[idx])


c = ax4.pcolormesh(cllon[:-1], cllat[:-1], xgrid, cmap="YlOrRd", vmin=0)
# c = ax4.pcolormesh(cllon[:-1], cllat[:-1], xgrid/tareas*1e6, cmap='YlOrRd', vmin=0, vmax=np.max(xgrid))
fig.colorbar(c, ax=ax4, fraction=0.03)
us.plot(ax=ax4, facecolor="none", edgecolor="k", linewidth=3)
ax4.set_xlim((-107, -90))
ax4.set_ylim((24.5, 37.5))
ax4.set_title("Posterior(kg km$^{-2}$ h$^{-1}$): " + full_vars[idx])

# plt.show()
plt.savefig("./Figures/sector-based_" + full_vars[idx] + ".png")
plt.show()

In [147]:
# df_wm.plot(column="prior", figsize=(10, 10), alpha=0.3,cmap='YlOrRd',ax=ax1,vmin=0,vmax=np.max(pgrid))
df_wm.explore(
    column="prior",  # make choropleth based on "BoroName" column
    #     tooltip="BoroName",  # show "BoroName" value in tooltip (on hover)
    #     popup=True,  # show all values in popup (on click)
    #     tiles="CartoDB positron",  # use "CartoDB positron" tiles
    #     cmap="Set1",  # use "Set1" matplotlib colormap
    #     style_kwds=dict(color="black"),  # use black outline
)